# AHP（階層分析法）

**AHP（Analytic Hierarchy Process、階層分析法）** は、Saaty（1980）が提唱した多基準意思決定（Multi-Criteria Decision Making, MCDM）の手法で、複数の評価基準・選択肢を **一対比較（pairwise comparison）** によって数値化された重みに変換する。

[Bradley-Terryモデル](bradley_terry.ipynb)と同じく「項目同士をペアで比較して全体の重みを求める」という点で目的は共通するが、AHPは確率モデルではなく、**一対比較行列の固有値問題**として重みを解く決定論的な手法である点が異なる。

## 一対比較行列

$J$個の項目（属性・選択肢など）について、すべてのペア$(i,j)$に対して「$i$は$j$に比べてどのくらい重要か」を、Saatyの**9段階比較尺度**で回答者に評定させる。

| 評点 | 意味 |
|---|---|
| 1 | 同等に重要 |
| 3 | やや重要 |
| 5 | 重要 |
| 7 | 非常に重要 |
| 9 | 極めて重要 |
| 2, 4, 6, 8 | 上記の中間 |

回答をもとに、項目$i$が項目$j$よりどれだけ重要かを$a_{ij}$として$J \times J$の **一対比較行列** $A = (a_{ij})$を作る。  
なお、対角成分（同じ項目同士）は$a_{ii}=1$とし、逆向きの比較は$a_{ji} = 1/a_{ij}$という **逆数性（reciprocity）** を仮定する。  

$$
A =
\begin{pmatrix}
1        & a_{12}   & \cdots & a_{1J} \\
1/a_{12} & 1        & \cdots & a_{2J} \\
\vdots   & \vdots   & \ddots & \vdots \\
1/a_{1J} & 1/a_{2J} & \cdots & 1
\end{pmatrix}
$$

## 重みの推定：固有値法

回答者の判断が完全に一貫している（**推移律**：$a_{ij} \cdot a_{jk} = a_{ik}$が常に成り立つ）場合、$A$の要素は真の重み$w_i$を使って$a_{ij} = w_i / w_j$と書け、このとき$A$は

$$
A \mathbf{w} = J \mathbf{w}
$$

を満たす。つまり真の重みベクトル$\mathbf{w}$は、$A$の **最大固有値$\lambda_{\max}=J$に対応する固有ベクトル** に一致する。

実際の人間の判断は完全に一貫しているとは限らないため、Saatyは最大固有値$\lambda_{\max}$に対応する固有ベクトルを正規化したものを重みの推定値$\hat{\mathbf{w}}$として採用することを提案した。この場合$\lambda_{\max} \ge J$となる。

## 整合度（Consistency Ratio）

回答者の判断が推移律からどれだけ逸脱しているかを、**整合度（Consistency Ratio, CR）** で評価する。

$$
CI = \frac{\lambda_{\max} - J}{J - 1}
\qquad
CR = \frac{CI}{RI}
$$

$CI$は**整合性指標（Consistency Index）**、$RI$は行列サイズ$J$ごとに定められた**ランダム整合性指標（Random Index、乱数から作った行列の平均的な$CI$）**である。慣習的に$CR \le 0.1$であれば「十分に一貫した回答」とみなし、それを超える場合は一対比較をやり直すべきとされる。

| $J$  | 3    | 4    | 5    | 6    | 7    | 8    |
| ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| $RI$ | 0.58 | 0.90 | 1.12 | 1.24 | 1.32 | 1.41 |


## 実装例

4つの評価基準（価格・品質・デザイン・ブランド）について、回答者が一対比較を行ったとする。

In [ ]:
import numpy as np
import pandas as pd

criteria = ["価格", "品質", "デザイン", "ブランド"]

# 一対比較行列（行が列に対してどれだけ重要かを表す。例: 価格は品質の1/3倍重要 = 品質の方が3倍重要）
A = np.array([
    [1,   1/3, 3,   5  ],  # 価格
    [3,   1,   5,   7  ],  # 品質
    [1/3, 1/5, 1,   3  ],  # デザイン
    [1/5, 1/7, 1/3, 1  ],  # ブランド
])

df_A = pd.DataFrame(A, index=criteria, columns=criteria)
df_A.round(2)

,価格,品質,デザイン,ブランド
価格,1.00,0.33,3.00,5.0
品質,3.00,1.00,5.00,7.0
デザイン,0.33,0.20,1.00,3.0
ブランド,0.20,0.14,0.33,1.0


In [5]:
eigvals, eigvecs = np.linalg.eig(A)

# 最大固有値とそれに対応する固有ベクトル
idx_max = np.argmax(eigvals.real)
lambda_max = eigvals[idx_max].real
w = eigvecs[:, idx_max].real
w = w / w.sum()  # 正規化(合計1)

weights = pd.Series(w, index=criteria, name="weight").sort_values(ascending=False)
print(f"最大固有値 λ_max = {lambda_max:.4f}")
weights

最大固有値 λ_max = 4.1170


品質      0.565009
価格      0.262201
デザイン    0.117504
ブランド    0.055285
Name: weight, dtype: float64

In [4]:
J = len(criteria)
RI_table = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41}

CI = (lambda_max - J) / (J - 1)
CR = CI / RI_table[J]

print(f"CI = {CI:.4f}")
print(f"CR = {CR:.4f}  ({'整合的 (CR<=0.1)' if CR <= 0.1 else '再考が必要 (CR>0.1)'})")


CI = 0.0390
CR = 0.0433  (整合的 (CR<=0.1))


この例では「品質」の重みが最も高く、続いて「価格」「デザイン」「ブランド」の順になっている。$CR$が$0.1$以下であれば、回答者の一対比較は十分に一貫していると判断できる。

## 階層構造への拡張

AHPの名前の由来である「階層」は、評価基準自体が複数レベルの階層構造を持つ場合に、各階層内で一対比較を行い、階層ごとの重みを掛け合わせて最終的な選択肢の総合スコアを求める仕組みを指す。

1. 最上位に目的（ゴール）を置く
2. その下に評価基準（本章の例：価格・品質・デザイン・ブランド）を置き、基準間の一対比較で重み$w_k$を求める
3. さらにその下に選択肢（代替案）を置き、各基準ごとに選択肢間の一対比較を行って基準ごとのスコア$s_{jk}$を求める
4. 選択肢$j$の総合スコアは$\sum_k w_k \, s_{jk}$として計算する

## コンジョイント分析・Bradley-Terryとの使い分け

| | AHP | Bradley-Terry | コンジョイント分析（CBC） |
|---|---|---|---|
| データ | 一対比較の強度評定（1〜9段階） | ペア比較の勝敗（二値） | 複数選択肢からの選択 |
| 推定原理 | 固有値法（決定論的） | 最尤法（確率モデル） | 最尤法（確率モデル） |
| 統計的推測 | 不可（標準誤差なし） | 可能 | 可能 |
| 適した場面 | 少数の基準・選択肢を専門家が評定 | 多数の項目を大量の比較データから評価 | 属性の組み合わせで構成される製品・サービスの評価 |

少数の評価基準に対して意思決定者自身が重みを内省的に判断したい場合はAHPが、多数の項目を多くの回答者からの比較データで統計的に評価したい場合はBradley-Terryやコンジョイント分析が適している。

## 参考

- [階層分析法「AHP」の考え方とPythonによる実装 | Logics of Blue](https://logics-of-blue.com/ahp-concept-and-implementation-by-python/)